# PageIndex — Step-by-Step Walkthrough

This notebook imports the actual functions from `pageindex` and walks through
the full PDF-to-tree pipeline one step at a time. For every LLM-driven step,
we also display the **prompt string** that the function sends to the model,
so you can see exactly what the LLM is asked to do.

## Pipeline summary

```
PDF  ──▶ [1] get_page_tokens
         ──▶ [2] find_toc_pages   (uses toc_detector_single_page)
             ──▶ [3] toc_extractor (uses detect_page_index)
                 ──▶ [4] check_toc → mode dispatch
                     ├─ Mode 1: process_toc_with_page_numbers
                     │     toc_transformer → toc_index_extractor
                     │     → calculate_page_offset → add_page_offset_to_toc_json
                     ├─ Mode 2: process_toc_no_page_numbers
                     │     toc_transformer → add_page_number_to_toc (per chunk)
                     └─ Mode 3: process_no_toc
                           generate_toc_init → generate_toc_continue (per chunk)
                 ──▶ [5] verify_toc + fix_incorrect_toc_with_retries
                     ──▶ [6] post_processing → list_to_tree
                         ──▶ [7] process_large_node_recursively
                             ──▶ [8] write_node_id → add_node_text
                                 → generate_summaries_for_structure
                                 → generate_doc_description
                                 ──▶ FINAL JSON TREE
```

> Every function in this notebook is imported from the real `pageindex`
> package at `https://github.com/VectifyAI/PageIndex`. Line references are
> against the `main` branch.

---
## Step 0 — Setup

### 0.1 Install dependencies

Run this once. The package is not on PyPI; we clone the repo and install
its `requirements.txt`.

> **Heads up — dependency conflict.** The repo's `requirements.txt` pins
> `python-dotenv==1.2.2`, but `litellm==1.83.7` requires `python-dotenv<1.1`,
> so `pip install -r requirements.txt` fails with `ResolutionImpossible`.
> Workaround: install the packages directly without that pin —
> `pip install litellm==1.83.7 pymupdf==1.26.4 PyPDF2==3.0.1 pyyaml==6.0.2`.
> This pulls `python-dotenv 1.0.1` (litellm-compatible); the basic
> `load_dotenv()` API PageIndex uses is unchanged.

In [8]:
# One-time setup. Skip if already done.
# !git clone https://github.com/VectifyAI/PageIndex.git
# %cd PageIndex
# requirements.txt overpins python-dotenv (see note above); install directly:
# !pip install -q litellm==1.83.7 pymupdf==1.26.4 PyPDF2==3.0.1 pyyaml==6.0.2

# If you already cloned, just point sys.path at the repo:
import sys, os
REPO_ROOT = os.path.abspath("../sample_pdfs")  # <-- adjust if your clone lives elsewhere
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)


### 0.2 Set your LLM API key

PageIndex uses [LiteLLM](https://docs.litellm.ai/) under the hood, so the
key name follows whichever provider you use. For OpenAI:

In [ ]:
import os
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Or, if you have a .env file in the repo root, the package will pick it up
# automatically via python-dotenv.


### 0.3 Pick a sample PDF

The repo ships with example PDFs in `sample_pdfs/`. We'll use one.

In [9]:
REPO_ROOT

'/Users/gieunkwak/Data_Analytics/SlowAI/rag-engineering-cookbook/sample_pdfs'

In [10]:
PDF_PATH = os.path.join(REPO_ROOT, "plos_sample.pdf")
# If that file isn't present, list what's available:
import glob
print(glob.glob(os.path.join(REPO_ROOT, "*.pdf"))[:5])
print("Using:", PDF_PATH)

['/Users/gieunkwak/Data_Analytics/SlowAI/rag-engineering-cookbook/sample_pdfs/plos_sample.pdf', '/Users/gieunkwak/Data_Analytics/SlowAI/rag-engineering-cookbook/sample_pdfs/frb_financial_stability_report.pdf']
Using: /Users/gieunkwak/Data_Analytics/SlowAI/rag-engineering-cookbook/sample_pdfs/plos_sample.pdf


### 0.4 Build the `opt` config object

`opt` is a SimpleNamespace-like object with all knobs the pipeline reads
(`opt.model`, `opt.toc_check_page_num`, `opt.max_page_num_each_node`,
`opt.max_token_num_each_node`, plus `if_add_*` flags). `ConfigLoader`
merges your overrides with defaults from `pageindex/config.yaml`.

In [11]:
from pageindex.utils import ConfigLoader

user_opt = {
    "model": "gpt-5.4-mini-2026-03-17",
    "toc_check_page_num": 20,
    "max_page_num_each_node": 10,
    "max_token_num_each_node": 20000,
    "if_add_node_id": "yes",
    "if_add_node_summary": "yes",
    "if_add_doc_description": "yes",
    "if_add_node_text": "no",
}
opt = ConfigLoader().load(user_opt)

# A simple console logger so the pipeline's logger.info(...) calls print:
import logging
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger("pageindex_walkthrough")

print("Model:        ", opt.model)
print("TOC check pages:", opt.toc_check_page_num)
print("Max pg/node:   ", opt.max_page_num_each_node)
print("Max tok/node:  ", opt.max_token_num_each_node)


Model:         gpt-5.4-mini-2026-03-17
TOC check pages: 20
Max pg/node:    10
Max tok/node:   20000


---
## Step 1 — Parse the PDF into `(text, token_count)` per page

**Function:** `get_page_tokens(pdf_path, model, pdf_parser)`
**File:** `pageindex/utils.py:413-437`

This is the only deterministic step in the whole pipeline — no LLM is called.
It opens the PDF (PyMuPDF / pdfplumber), extracts text page by page, and
counts tokens via `litellm.token_counter`. The returned `page_list` is the
single source of truth that flows through every subsequent step.

In [12]:
from pageindex.utils import get_page_tokens, count_tokens

page_list = get_page_tokens(PDF_PATH, model=opt.model)

print(f"Total pages: {len(page_list)}")
print(f"Each entry is a tuple: (page_text, token_count)")
print()
print("--- First page preview ---")
print(f"Text  (first 300 chars): {page_list[0][0][:300]!r}")
print(f"Tokens: {page_list[0][1]}")
print()
print(f"Total tokens in document: {sum(t for _, t in page_list):,}")


Total pages: 12
Each entry is a tuple: (page_text, token_count)

--- First page preview ---
Text  (first 300 chars): 'PLOS One | https://doi.org/10.1371/journal.pone.0349108  May 7, 2026 1 / 12\n \n OPEN ACCESS\nCitation: Nakanishi T, Tani A, Matsumoto K, \nIshikawa A, Takami Y, Kamimura Y, et al. (2026) \nClinical evaluation of the i-gel Plus supraglottic \nairway in Japanese patients: A prospective \nobservational study'
Tokens: 815

Total tokens in document: 11,847


**Why token counts matter.** The pipeline uses tokens to (a) decide when a
chunk needs to be split before sending to the LLM, and (b) decide when a
leaf node in the final tree needs to be split *recursively* via
`process_large_node_recursively`.

---
## Step 2 — Find which pages contain the Table of Contents

**Functions:**
- `find_toc_pages(start_page_index, page_list, opt, logger)` — `page_index.py:333-358`
- `toc_detector_single_page(content, model)` — `page_index.py:104-123`

`find_toc_pages` walks pages 0..`opt.toc_check_page_num`, calling
`toc_detector_single_page` on each. As soon as it sees a TOC page followed
by a non-TOC page, it stops and returns the contiguous block.

### The prompt used by `toc_detector_single_page`

This is the exact prompt string the function sends to the LLM:

In [13]:
TOC_DETECTOR_PROMPT = """
Your job is to detect if there is a table of content provided in the given text.

Given text: {content}

return the following JSON format:
{{
    "thinking": <why do you think there is a table of content in the given text>
    "toc_detected": "<yes or no>",
}}
Directly return the final JSON structure. Do not output anything else.

Please note: abstract,summary, notation list, figure list, table list, etc. are not table of contents."""

print(TOC_DETECTOR_PROMPT)



Your job is to detect if there is a table of content provided in the given text.

Given text: {content}

return the following JSON format:
{{
    "thinking": <why do you think there is a table of content in the given text>
    "toc_detected": "<yes or no>",
}}
Directly return the final JSON structure. Do not output anything else.

Please note: abstract,summary, notation list, figure list, table list, etc. are not table of contents.


In [18]:
from pageindex.page_index import find_toc_pages, toc_detector_single_page

# Run the detector on a single page first, just to see it work in isolation:
verdict_page0 = toc_detector_single_page(page_list[0][0], model=opt.model)
print(f"Page 0 toc_detected: {verdict_page0!r}")

23:14:13 - LiteLLM:INFO: utils.py:4007 - 
LiteLLM completion() model= gpt-5.4-mini-2026-03-17; provider = openai
[INFO] 
LiteLLM completion() model= gpt-5.4-mini-2026-03-17; provider = openai
23:14:14 - LiteLLM:INFO: utils.py:1650 - Wrapper: Completed Call, calling success_handler
[INFO] Wrapper: Completed Call, calling success_handler


Page 0 toc_detected: 'no'


In [ ]:
# Now run the full search:
toc_page_list = find_toc_pages(
    start_page_index=0,
    page_list=page_list,
    opt=opt,
    logger=logger,
)
print(f"TOC pages found: {toc_page_list}")

---
## Step 3 — Extract the TOC text and check whether it has page numbers

**Functions:**
- `toc_extractor(page_list, toc_page_list, model)` — `page_index.py:219-235`
- `detect_page_index(toc_content, model)` — `page_index.py:199-217`

`toc_extractor` concatenates the text of every TOC page, normalizes leader
dots (`...`) to colons, then asks the LLM whether the TOC contains page
numbers. The answer determines which **mode** Step 4 dispatches to.

### The prompt used by `detect_page_index`

In [15]:
DETECT_PAGE_INDEX_PROMPT = """
You will be given a table of contents.
Your job is to detect if there are page numbers/indices given within the table of contents.

Given text: {toc_content}

Reply format:
{{
    "thinking": <why do you think there are page numbers/indices given within the table of contents>
    "page_index_given_in_toc": "<yes or no>"
}}
Directly return the final JSON structure. Do not output anything else."""

print(DETECT_PAGE_INDEX_PROMPT)



You will be given a table of contents.
Your job is to detect if there are page numbers/indices given within the table of contents.

Given text: {toc_content}

Reply format:
{{
    "thinking": <why do you think there are page numbers/indices given within the table of contents>
    "page_index_given_in_toc": "<yes or no>"
}}
Directly return the final JSON structure. Do not output anything else.


In [16]:
from pageindex.page_index import toc_extractor, detect_page_index

toc_result = toc_extractor(page_list, toc_page_list, opt.model)

print(f"page_index_given_in_toc: {toc_result['page_index_given_in_toc']!r}")
print()
print("--- TOC content preview (first 800 chars) ---")
print(toc_result["toc_content"][:800])


23:11:36 - LiteLLM:INFO: utils.py:4007 - 
LiteLLM completion() model= gpt-5.4-mini-2026-03-17; provider = openai
[INFO] 
LiteLLM completion() model= gpt-5.4-mini-2026-03-17; provider = openai


start detect_page_index


23:11:37 - LiteLLM:INFO: utils.py:1650 - Wrapper: Completed Call, calling success_handler
[INFO] Wrapper: Completed Call, calling success_handler


page_index_given_in_toc: 'no'

--- TOC content preview (first 800 chars) ---



---
## Step 4 — Mode dispatch via `check_toc`

**Function:** `check_toc(page_list, opt)` — `page_index.py:688-725`

This wraps Steps 2–3 and decides which of three modes to run:

| Mode | Triggered when | Handler |
|---|---|---|
| 1 | TOC found AND has page numbers | `process_toc_with_page_numbers` |
| 2 | TOC found BUT no page numbers   | `process_toc_no_page_numbers`  |
| 3 | No TOC found                    | `process_no_toc`               |

In [17]:
from pageindex.page_index import check_toc

check_result = check_toc(page_list, opt=opt)
print("toc_page_list:           ", check_result["toc_page_list"])
print("page_index_given_in_toc: ", check_result["page_index_given_in_toc"])

# Decide the mode:
if check_result["toc_content"] is None:
    MODE = "process_no_toc"
elif check_result["page_index_given_in_toc"] == "yes":
    MODE = "process_toc_with_page_numbers"
else:
    MODE = "process_toc_no_page_numbers"
print(f"\n>>> MODE = {MODE}")


23:13:24 - LiteLLM:INFO: utils.py:4007 - 
LiteLLM completion() model= gpt-5.4-mini-2026-03-17; provider = openai
[INFO] 
LiteLLM completion() model= gpt-5.4-mini-2026-03-17; provider = openai


start find_toc_pages


23:13:25 - LiteLLM:INFO: utils.py:1650 - Wrapper: Completed Call, calling success_handler
[INFO] Wrapper: Completed Call, calling success_handler
23:13:25 - LiteLLM:INFO: utils.py:4007 - 
LiteLLM completion() model= gpt-5.4-mini-2026-03-17; provider = openai
[INFO] 
LiteLLM completion() model= gpt-5.4-mini-2026-03-17; provider = openai
23:13:26 - LiteLLM:INFO: utils.py:1650 - Wrapper: Completed Call, calling success_handler
[INFO] Wrapper: Completed Call, calling success_handler
23:13:26 - LiteLLM:INFO: utils.py:4007 - 
LiteLLM completion() model= gpt-5.4-mini-2026-03-17; provider = openai
[INFO] 
LiteLLM completion() model= gpt-5.4-mini-2026-03-17; provider = openai
23:13:30 - LiteLLM:INFO: utils.py:1650 - Wrapper: Completed Call, calling success_handler
[INFO] Wrapper: Completed Call, calling success_handler
23:13:30 - LiteLLM:INFO: utils.py:4007 - 
LiteLLM completion() model= gpt-5.4-mini-2026-03-17; provider = openai
[INFO] 
LiteLLM completion() model= gpt-5.4-mini-2026-03-17; prov

no toc found
toc_page_list:            []
page_index_given_in_toc:  no

>>> MODE = process_no_toc


---
## Step 5A — Mode 1: TOC **with** page numbers

**Function:** `process_toc_with_page_numbers()` — `page_index.py:614-643`

Four sub-steps. Run this section if `MODE == "process_toc_with_page_numbers"`.

### 5A.1 — `toc_transformer`: raw TOC text → JSON list

**File:** `page_index.py:270-328`

Converts the raw extracted TOC text into a list of
`{"structure": "1.1.2", "title": "...", "page": 42}` records.

#### Prompt for `toc_transformer`

In [ ]:
TOC_TRANSFORMER_PROMPT = """
You are given a table of contents, You job is to transform the whole table of content into a JSON format included table_of_contents.
structure is the numeric system which represents the index of the hierarchy section in the table of contents. For example, the first section has structure index 1, the first subsection has structure index 1.1, the second subsection has structure index 1.2, etc.
The response should be in the following JSON format:
{
    table_of_contents: [
        {
            "structure": <structure index, "x.x.x" or None> (string),
            "title": <title of the section>,
            "page": <page number or None>,
        },
        ...
    ],
}
You should transform the full table of contents in one go.
Directly return the final JSON structure, do not output anything else. """

print(TOC_TRANSFORMER_PROMPT)


In [ ]:
from pageindex.page_index import toc_transformer

toc_with_page_number = toc_transformer(check_result["toc_content"], opt.model)

print(f"Got {len(toc_with_page_number)} TOC entries.")
print()
print("--- First 5 entries ---")
import pprint; pprint.pp(toc_with_page_number[:5])


### 5A.2 — `toc_index_extractor`: match printed page numbers to physical indices

**File:** `page_index.py:240-266`

The TOC says "Chapter 3 — page 47". But "page 47" might mean physical page 51
because the front matter (cover, copyright, TOC itself) has been numbered
with roman numerals. This function reads pages right after the TOC, wraps
them in `<physical_index_X>...<physical_index_X>` tags, and asks the LLM to
locate each section title.

#### Prompt for `toc_index_extractor`

In [ ]:
TOC_INDEX_EXTRACTOR_PROMPT = """
You are given a table of contents in a json format and several pages of a document, your job is to add the physical_index to the table of contents in the json format.

The provided pages contains tags like <physical_index_X> and <physical_index_X> to indicate the physical location of the page X.
The structure variable is the numeric system which represents the index of the hierarchy section in the table of contents. For example, the first section has structure index 1, the first subsection has structure index 1.1, the second subsection has structure index 1.2, etc.

The response should be in the following JSON format:
[
    {
        "structure": <structure index, "x.x.x" or None> (string),
        "title": <title of the section>,
        "physical_index": "<physical_index_X>" (keep the format)
    },
    ...
]

Only add the physical_index to the sections that are in the provided pages.
If the section is not in the provided pages, do not add the physical_index to it.

Directly return the final JSON structure. Do not output anything else."""

print(TOC_INDEX_EXTRACTOR_PROMPT)


In [ ]:
from pageindex.page_index import (
    toc_index_extractor,
    remove_page_number,
    extract_matching_page_pairs,
)
from pageindex.utils import convert_physical_index_to_int
import copy

# Build a copy of the TOC list with the page numbers stripped (the LLM should
# fill them back in based on what it sees in the actual pages):
toc_no_page_number = remove_page_number(copy.deepcopy(toc_with_page_number))

# Wrap the pages right after the TOC with <physical_index_N> tags:
start_page_index = toc_page_list[-1] + 1
main_content = ""
for page_index in range(start_page_index, min(start_page_index + opt.toc_check_page_num, len(page_list))):
    main_content += f"<physical_index_{page_index+1}>\n{page_list[page_index][0]}\n<physical_index_{page_index+1}>\n\n"

toc_with_physical_index = toc_index_extractor(toc_no_page_number, main_content, opt.model)
toc_with_physical_index = convert_physical_index_to_int(toc_with_physical_index)

print("--- Sample physical-index assignments ---")
import pprint; pprint.pp(toc_with_physical_index[:5])


### 5A.3 — `calculate_page_offset`: figure out the front-matter offset

**File:** `page_index.py:386-406`

For each section that appears in BOTH lists (with-printed-page and
with-physical-index), compute `physical_index - page`. The most common
difference is the front-matter offset.

In [ ]:
from pageindex.page_index import calculate_page_offset

matching_pairs = extract_matching_page_pairs(
    toc_with_page_number, toc_with_physical_index, start_page_index
)
print(f"Found {len(matching_pairs)} matching pairs.")
import pprint; pprint.pp(matching_pairs[:5])

offset = calculate_page_offset(matching_pairs)
print(f"\n>>> Most-common offset: {offset}")
print("    (i.e. printed page N corresponds to physical page N+%d)" % (offset or 0))


### 5A.4 — `add_page_offset_to_toc_json`: apply the offset to every entry

**File:** `page_index.py:408-414`

In [ ]:
from pageindex.page_index import add_page_offset_to_toc_json

toc_with_page_number = add_page_offset_to_toc_json(toc_with_page_number, offset)
print("--- After offset applied ---")
import pprint; pprint.pp(toc_with_page_number[:5])


---
## Step 5B — Mode 2: TOC **without** page numbers

**Function:** `process_toc_no_page_numbers()` — `page_index.py:589-610`

When the TOC exists but has no page numbers, we can't compute an offset.
Instead, we walk the document in chunks and ask the LLM "does the next
section start in this chunk?" for each entry.

> Run this only if `MODE == "process_toc_no_page_numbers"`. We show the
> prompt regardless so you can see how it works.

### Prompt for `add_page_number_to_toc` (page_index.py:453-483)

In [ ]:
ADD_PAGE_NUMBER_TO_TOC_PROMPT = """
You are given an JSON structure of a document and a partial part of the document. Your task is to check if the title that is described in the structure is started in the partial given document.

The provided text contains tags like <physical_index_X> and <physical_index_X> to indicate the physical location of the page X.

If the full target section starts in the partial given document, insert the given JSON structure with the "start": "yes", and "start_index": "<physical_index_X>".
If the full target section does not start in the partial given document, insert "start": "no",  "start_index": None.

The response should be in the following format.
[
    {
        "structure": <structure index, "x.x.x" or None> (string),
        "title": <title of the section>,
        "start": "<yes or no>",
        "physical_index": "<physical_index_X> (keep the format)" or None
    },
    ...
]
The given structure contains the result of the previous part, you need to fill the result of the current part, do not change the previous result.
Directly return the final JSON structure. Do not output anything else."""

print(ADD_PAGE_NUMBER_TO_TOC_PROMPT)


In [ ]:
# Reference only — runs the full Mode 2 path if your document needs it:
if MODE == "process_toc_no_page_numbers":
    from pageindex.page_index import process_toc_no_page_numbers
    toc_with_page_number = process_toc_no_page_numbers(
        check_result["toc_content"],
        toc_page_list,
        page_list,
        model=opt.model,
        logger=logger,
    )
    import pprint; pprint.pp(toc_with_page_number[:5])
else:
    print(f"(skipping — MODE is {MODE})")


---
## Step 5C — Mode 3: No TOC at all

**Function:** `process_no_toc()` — `page_index.py:568-587`

When there's no TOC, we have to *invent* one. The doc is chunked into
groups of ~20k tokens; the first chunk is sent to `generate_toc_init`,
each subsequent chunk to `generate_toc_continue`.

### Prompt for `generate_toc_init` (page_index.py:534-566)

In [ ]:
GENERATE_TOC_INIT_PROMPT = """
You are an expert in extracting hierarchical tree structure, your task is to generate the tree structure of the document.
The structure variable is the numeric system which represents the index of the hierarchy section in the table of contents. For example, the first section has structure index 1, the first subsection has structure index 1.1, the second subsection has structure index 1.2, etc.
For the title, you need to extract the original title from the text, only fix the space inconsistency.
The provided text contains tags like <physical_index_X> and <physical_index_X> to indicate the start and end of page X.
For the physical_index, you need to extract the physical index of the start of the section from the text. Keep the <physical_index_X> format.

The response should be in the following format.
[
    {{
        "structure": <structure index, "x.x.x"> (string),
        "title": <title of the section, keep the original title>,
        "physical_index": "<physical_index_X> (keep the format)"
    }},
],

Directly return the final JSON structure. Do not output anything else."""

print(GENERATE_TOC_INIT_PROMPT)


### Prompt for `generate_toc_continue` (page_index.py:499-531)

In [ ]:
GENERATE_TOC_CONTINUE_PROMPT = """
You are an expert in extracting hierarchical tree structure.
You are given a tree structure of the previous part and the text of the current part.
Your task is to continue the tree structure from the previous part to include the current part.

The structure variable is the numeric system which represents the index of the hierarchy section in the table of contents. For example, the first section has structure index 1, the first subsection has structure index 1.1, the second subsection has structure index 1.2, etc.
For the title, you need to extract the original title from the text, only fix the space inconsistency.
The provided text contains tags like <physical_index_X> and <physical_index_X> to indicate the start and end of page X.
For the physical_index, you need to extract the physical index of the start of the section from the text. Keep the <physical_index_X> format.

The response should be in the following format.
[
    {
        "structure": <structure index, "x.x.x"> (string),
        "title": <title of the section, keep the original title>,
        "physical_index": "<physical_index_X> (keep the format)"
    },
    ...
]

Directly return the additional part of the final JSON structure. Do not output anything else."""

print(GENERATE_TOC_CONTINUE_PROMPT)


In [ ]:
# Reference only — runs the full Mode 3 path if your document needs it:
if MODE == "process_no_toc":
    from pageindex.page_index import process_no_toc
    toc_with_page_number = process_no_toc(
        page_list, start_index=1, model=opt.model, logger=logger
    )
    import pprint; pprint.pp(toc_with_page_number[:5])
else:
    print(f"(skipping — MODE is {MODE})")


---
## Step 6 — Verify each section's physical index, then fix wrong ones

**Functions:**
- `validate_and_truncate_physical_indices()` — `page_index.py:1114-1144`
- `verify_toc()` — `page_index.py:892-944`
- `check_title_appearance()` — `page_index.py` (top of file)
- `fix_incorrect_toc_with_retries()` — `page_index.py:870-886`

`verify_toc` runs `check_title_appearance` concurrently for every entry,
asking the LLM "does this section title actually appear on this page?".
If accuracy > 0.6 but < 1.0, `fix_incorrect_toc_with_retries` searches the
range between known-correct anchors to relocate the wrong entries.

### Prompt for `check_title_appearance`

In [ ]:
CHECK_TITLE_APPEARANCE_PROMPT = """
Your job is to check if the given section appears or starts in the given page_text.

Note: do fuzzy matching, ignore any space inconsistency in the page_text.

The given section title is {title}.
The given page_text is {page_text}.

Reply format:
{{
    "thinking": <why do you think the section appears or starts in the page_text>
    "answer": "yes or no" (yes if the section appears or starts in the page_text, no otherwise)
}}
Directly return the final JSON structure. Do not output anything else."""

print(CHECK_TITLE_APPEARANCE_PROMPT)


In [ ]:
import asyncio
from pageindex.page_index import (
    validate_and_truncate_physical_indices,
    verify_toc,
    fix_incorrect_toc_with_retries,
)

# 1) drop entries with no physical_index, cap any out-of-range indices
toc_with_page_number = [item for item in toc_with_page_number if item.get("physical_index") is not None]
toc_with_page_number = validate_and_truncate_physical_indices(
    toc_with_page_number, len(page_list), start_index=1, logger=logger
)

# 2) verify (concurrent LLM calls)
accuracy, incorrect_results = asyncio.run(
    verify_toc(page_list, toc_with_page_number, start_index=1, model=opt.model)
)
print(f"Verification accuracy: {accuracy*100:.1f}%")
print(f"Incorrect entries:     {len(incorrect_results)}")

# 3) if some are wrong but most are right, repair them
if 0.6 < accuracy < 1.0 and incorrect_results:
    toc_with_page_number, still_wrong = asyncio.run(
        fix_incorrect_toc_with_retries(
            toc_with_page_number, page_list, incorrect_results,
            start_index=1, max_attempts=3, model=opt.model, logger=logger,
        )
    )
    print(f"After fix attempts, still wrong: {len(still_wrong)}")


**Failure-recovery logic** (from `meta_processor`, `page_index.py` ~line 760):
if accuracy is too low, the pipeline doesn't give up — it falls through to
the next mode. Mode 1 → Mode 2 → Mode 3.

---
## Step 7 — Convert the flat list into a hierarchical tree

**Functions:**
- `post_processing(structure, page_list, ...)` — `pageindex/utils.py:460-479`
- `list_to_tree(structure)` — `pageindex/utils.py:350-396`

`post_processing` fills in `start_index` and `end_index` for every node by
looking at where the *next* sibling starts. `list_to_tree` then nests
nodes by their dotted `structure` codes — `1.1.2` becomes a child of
`1.1`, which is a child of `1`.

In [ ]:
from pageindex.utils import post_processing, list_to_tree

# 1) compute (start_index, end_index) per node
toc_with_page_number = post_processing(toc_with_page_number, page_list)

print("--- After post_processing (first 3) ---")
import pprint; pprint.pp(toc_with_page_number[:3])


In [ ]:
# 2) nest into a tree
tree = list_to_tree(toc_with_page_number)

def show_tree(nodes, depth=0, max_depth=3):
    for n in nodes:
        title = n.get("title", "<no title>")
        s = n.get("start_index"); e = n.get("end_index")
        print(f"{'  ' * depth}- {title}  [pages {s}–{e}]")
        if depth < max_depth and "nodes" in n:
            show_tree(n["nodes"], depth + 1, max_depth)

show_tree(tree, max_depth=2)


---
## Step 8 — Recursively split oversized leaf nodes

**Function:** `process_large_node_recursively(node, page_list, opt, logger)`
**File:** `page_index.py:992-1019`

If a leaf node spans more than `opt.max_page_num_each_node` pages or holds
more than `opt.max_token_num_each_node` tokens, this function treats that
node's text slice as a brand-new mini-document and runs the entire
pipeline (Steps 2–7) on it, then attaches the resulting subtree.

This is what gives PageIndex arbitrary depth — it's the only step that
uses the package recursively.

In [ ]:
import asyncio
from pageindex.page_index import process_large_node_recursively

async def expand_all_oversized_leaves(tree_nodes):
    for node in tree_nodes:
        if "nodes" in node and node["nodes"]:
            await expand_all_oversized_leaves(node["nodes"])
        else:
            await process_large_node_recursively(node, page_list, opt=opt, logger=logger)

asyncio.run(expand_all_oversized_leaves(tree))
print("Recursive expansion done.")
show_tree(tree, max_depth=3)


---
## Step 9 — Enrichment: IDs, text, summaries, doc description

The tree skeleton is now correct. The remaining steps annotate it. Each
is gated by an `opt.if_add_*` flag.

### 9.1 — `write_node_id`: assign 4-digit IDs in DFS order

**File:** `pageindex/utils.py`

Walks the tree in depth-first order, assigning `0000`, `0001`, `0002`, ...
These IDs are what your retrieval LLM will return when it picks nodes.

In [ ]:
from pageindex.utils import write_node_id

if opt.if_add_node_id == "yes":
    write_node_id(tree, start=0)

# Show the IDs:
def show_ids(nodes, depth=0, max_depth=2):
    for n in nodes:
        print(f"{'  ' * depth}{n.get('node_id', '----')} {n.get('title', '')}")
        if depth < max_depth and "nodes" in n:
            show_ids(n["nodes"], depth + 1, max_depth)

show_ids(tree, max_depth=2)


### 9.2 — `add_node_text`: attach raw page text to every node

**File:** `pageindex/utils.py:579-589`

Slices `page_list[start_index-1:end_index]` and stores it on the node as
the `text` field. Only emitted in the final JSON if `opt.if_add_node_text == "yes"`.

This is also a prerequisite for generating summaries in 9.3.

In [ ]:
from pageindex.utils import add_node_text

add_node_text(tree, page_list)
print("Sample node 'text' (first 200 chars):")
def first_leaf(nodes):
    for n in nodes:
        if "nodes" not in n or not n["nodes"]:
            return n
        return first_leaf(n["nodes"])
leaf = first_leaf(tree)
print(leaf["text"][:200])


### 9.3 — `generate_summaries_for_structure`: one summary per node

**File:** `pageindex/utils.py:616-623`

Walks the tree and asks the LLM for a 1–3 sentence summary per node.
Leaves get summarized from their full text; non-leaf nodes get
summarized from their children's summaries (cheaper + keeps the
hierarchy coherent).

The prompts live in `pageindex/utils.py` (around lines 600-615). They
look like this in spirit (paraphrased from the package source):

In [ ]:
NODE_SUMMARY_PROMPT_LEAF = """
You are given the text content of a section of a document.
Your task is to generate a concise summary of the section, around 1 to 3 sentences.

Section title: {title}
Section text:
{text}

Directly return the summary, do not include any other text."""

NODE_SUMMARY_PROMPT_PARENT = """
You are given the title of a section and the summaries of its subsections.
Your task is to generate a concise summary of this section based on its subsections, around 1 to 3 sentences.

Section title: {title}
Subsection summaries:
{children_summaries}

Directly return the summary, do not include any other text."""

# (These mirror the structure of the actual prompts in pageindex/utils.py.
#  Inspect the live source to see the exact wording in your version.)
print("Leaf prompt template:\n", NODE_SUMMARY_PROMPT_LEAF)
print("\nParent prompt template:\n", NODE_SUMMARY_PROMPT_PARENT)


In [ ]:
import asyncio
from pageindex.utils import generate_summaries_for_structure

if opt.if_add_node_summary == "yes":
    asyncio.run(generate_summaries_for_structure(tree, model=opt.model))

# Show:
print("--- Summaries on first few nodes ---")
for n in tree[:3]:
    print(f"[{n.get('node_id')}] {n.get('title')}")
    print(f"  → {n.get('summary', '(no summary)')[:200]}")
    print()


### 9.4 — `generate_doc_description`: one sentence for the whole document

**File:** `pageindex/utils.py:605-613`

Built from the tree-of-summaries (no full text), so the prompt stays
small even for huge documents. The output is what powers
*Search-by-Description* in the retrieval side of PageIndex.

In [ ]:
DOC_DESCRIPTION_PROMPT = """
You are given a table of contents structure of a document.
Your task is to generate a one-sentence description for the document
that makes it easy to distinguish from other documents.

Document tree structure:
{PageIndex_Tree}

Directly return the description, do not include any other text."""

print(DOC_DESCRIPTION_PROMPT)


In [ ]:
from pageindex.utils import generate_doc_description

if opt.if_add_doc_description == "yes":
    doc_description = asyncio.run(generate_doc_description(tree, model=opt.model))
    print("Document description:")
    print(doc_description)


---
## Final output

Everything in `tree` plus the doc description. This is the JSON
`page_index_main` would have produced if we'd run it end-to-end.

In [ ]:
import json

final_output = {
    "doc_description": doc_description if opt.if_add_doc_description == "yes" else None,
    "structure": tree,
}

# Optional: drop the heavy `text` field if we set if_add_node_text="no"
if opt.if_add_node_text != "yes":
    def strip_text(nodes):
        for n in nodes:
            n.pop("text", None)
            if "nodes" in n:
                strip_text(n["nodes"])
    strip_text(final_output["structure"])

# Save
out_path = "./pageindex_walkthrough_result.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)
print(f"Wrote {out_path}")

# Preview
print(json.dumps(final_output["structure"][:2], indent=2)[:1500])


---
## Sanity check — compare against the one-shot API

You should get the same (or very nearly the same — the LLM is non-deterministic)
result by calling the public entry point directly:

```python
from pageindex import page_index
result = page_index(PDF_PATH, **user_opt)
```

Or `page_index_main(PDF_PATH, opt)` if you've already built `opt`.

If your result here diverges materially, the most likely culprits are
non-determinism in the verification step (Step 6) or a different
front-matter offset getting picked up in 5A.3.

---

## Function index (for quick lookup)

| Step | Function | File:lines |
|---|---|---|
| 1 | `get_page_tokens`                        | `utils.py:413-437`        |
| 1 | `count_tokens`                           | `utils.py:25-28`          |
| 2 | `find_toc_pages`                         | `page_index.py:333-358`   |
| 2 | `toc_detector_single_page`               | `page_index.py:104-123`   |
| 3 | `toc_extractor`                          | `page_index.py:219-235`   |
| 3 | `detect_page_index`                      | `page_index.py:199-217`   |
| 4 | `check_toc`                              | `page_index.py:688-725`   |
| 5A | `process_toc_with_page_numbers`         | `page_index.py:614-643`   |
| 5A | `toc_transformer`                       | `page_index.py:270-328`   |
| 5A | `toc_index_extractor`                   | `page_index.py:240-266`   |
| 5A | `calculate_page_offset`                 | `page_index.py:386-406`   |
| 5A | `add_page_offset_to_toc_json`           | `page_index.py:408-414`   |
| 5B | `process_toc_no_page_numbers`           | `page_index.py:589-610`   |
| 5B | `add_page_number_to_toc`                | `page_index.py:453-483`   |
| 5C | `process_no_toc`                        | `page_index.py:568-587`   |
| 5C | `generate_toc_init`                     | `page_index.py:534-566`   |
| 5C | `generate_toc_continue`                 | `page_index.py:499-531`   |
| 6 | `validate_and_truncate_physical_indices` | `page_index.py:1114-1144` |
| 6 | `verify_toc`                             | `page_index.py:892-944`   |
| 6 | `check_title_appearance`                 | `page_index.py` (top)     |
| 6 | `fix_incorrect_toc_with_retries`         | `page_index.py:870-886`   |
| 7 | `post_processing`                        | `utils.py:460-479`        |
| 7 | `list_to_tree`                           | `utils.py:350-396`        |
| 8 | `process_large_node_recursively`         | `page_index.py:992-1019`  |
| 9 | `write_node_id`                          | `utils.py`                |
| 9 | `add_node_text`                          | `utils.py:579-589`        |
| 9 | `generate_summaries_for_structure`       | `utils.py:616-623`        |
| 9 | `generate_doc_description`               | `utils.py:605-613`        |
| — | `page_index_main` (orchestrator)         | `page_index.py:1058-1101` |
| — | `page_index` (public API)                | `page_index.py:1103-1111` |